In [4]:
import sys
import os

In [5]:
# Adds the project root directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [13]:
import pandas as pd

In [11]:
from src.preprocessing.feature_engineering import  FeatureEngineeringHandler

In [15]:
df = pd.read_csv("../data/interim/kickstarter_cleaned.csv")

In [17]:


fe = FeatureEngineeringHandler()

df_fe = fe.fit_transform(df)

In [19]:
df_fe.to_csv("../data/interim/kickstarter_featured.csv", index=False)

df_fe.head()

,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,...,name_word_count,avg_word_length,uppercase_ratio,goal_per_day,goal_duration_interaction,goal_month_interaction,pledged_log,pledge_ratio,is_overfunded,log_goal_pledge_gap
0,1.000002e+09,The Songs of Adelaide & Abullah,Poetry,Publishing,GBP,2015-10-09,1000.0,2015-08-11 12:12:28,0.0,failed,...,6,5.166667,0.129032,17.241379,400.707777,55.270038,0.000000,0.000000,0,6.908755
1,1.000004e+09,Greeting From Earth: ZGAC Arts Capsule For ET,Narrative Film,Film & Video,USD,2017-11-01,30000.0,2017-09-02 04:43:57,2421.0,failed,...,8,5.625000,0.266667,508.474576,608.230174,92.780874,7.792349,0.080700,0,2.516637
2,1.000004e+09,Where is Hank?,Narrative Film,Film & Video,USD,2013-02-26,45000.0,2013-01-12 00:20:50,220.0,failed,...,3,4.666667,0.142857,1022.727273,471.435360,10.714440,5.398163,0.004889,0,5.316277
3,1.000008e+09,ToshiCapital Rekordz Needs Help to Complete Album,Music,Music,USD,2012-04-16,5000.0,2012-03-17 03:24:11,1.0,failed,...,7,7.000000,0.142857,172.413793,247.004402,25.552180,0.693147,0.000200,0,7.824246
4,1.000011e+09,Community Film Project: The Art of Neighborhoo...,Film & Video,Film & Video,USD,2015-08-29,19500.0,2015-07-04 08:35:03,1283.0,canceled,...,8,7.250000,0.120690,354.545455,543.302156,69.147547,7.157735,0.065795,0,2.720486


In [20]:
from src.preprocessing.feature_engineering import FeatureEngineeringHandler

from src.preprocessing.encoding import EncodingHandler

In [ ]:


# df = pd.read_csv("../data/interim/kickstarter_featured.csv")

In [ ]:
# fe = FeatureEngineeringHandler()

# df = fe.fit_transform(df)

In [ ]:
# encoder = EncodingHandler(

#     encoding_mapping={

#         "country": "onehot",

#         "category": "onehot"   # or "label" if you want

#     }
# # 
# )

In [40]:
import pandas as pd

from src.preprocessing.feature_engineering import FeatureEngineeringHandler
from src.preprocessing.encoding import EncodingHandler


# -------------------------
# Load data
# -------------------------
df = pd.read_csv("../data/interim/kickstarter_featured.csv")


# -------------------------
# Feature Engineering
# -------------------------
fe = FeatureEngineeringHandler()
df = fe.fit_transform(df)


# -------------------------
# FIX: define correct target column
# -------------------------
print("Columns:", df.columns)


# -------------------------
# Use correct target
# -------------------------
target_col = "state"   # VERY likely correct for Kickstarter

X = df.drop(columns=[target_col])
y = df[target_col]


# -------------------------
# Encoding
# -------------------------
encoder = EncodingHandler(
    encoding_mapping={
        "country": "onehot",
        "category": "label"
    }
)

encoder.fit(X, y)
X_encoded = encoder.transform(X)


print(X_encoded.head())

Columns: Index(['ID', 'name', 'category', 'main_category', 'currency', 'deadline',
       'goal', 'launched', 'pledged', 'state', 'backers', 'country',
       'usd pledged', 'usd_pledged_real', 'usd_goal_real', 'campaign_duration',
       'launch_year', 'launch_month', 'launch_weekday', 'launch_quarter',
       'is_weekend_launch', 'goal_log', 'is_high_goal', 'name_length',
       'name_word_count', 'avg_word_length', 'uppercase_ratio', 'goal_per_day',
       'goal_duration_interaction', 'goal_month_interaction', 'pledged_log',
       'pledge_ratio', 'is_overfunded', 'log_goal_pledge_gap'],
      dtype='str')
             ID                                               name  category  \
0  1.000002e+09                    The Songs of Adelaide & Abullah       108   
1  1.000004e+09      Greeting From Earth: ZGAC Arts Capsule For ET        93   
2  1.000004e+09                                     Where is Hank?        93   
3  1.000008e+09  ToshiCapital Rekordz Needs Help to Complete Al

In [45]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.preprocessing.feature_engineering import FeatureEngineeringHandler
from src.preprocessing.encoding import EncodingHandler
from src.preprocessing.scaling import ScalingHandler


# -------------------------
# Load Data
# -------------------------
df = pd.read_csv("../data/interim/kickstarter_featured.csv")


# -------------------------
# Feature Engineering
# -------------------------
fe = FeatureEngineeringHandler()
df = fe.fit_transform(df)


# -------------------------
# Define target
# -------------------------
target_col = "state"

X = df.drop(columns=[target_col])
y = df[target_col]


# -------------------------
# TRAIN - TEST SPLIT (IMPORTANT FIX)
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =================================================
# 1. FEATURE ENGINEERING already done above
# =================================================


# =================================================
# 2. ENCODING (FIT ONLY ON TRAIN → NO LEAKAGE)
# =================================================

encoder = EncodingHandler(
    encoding_mapping={
        "country": "onehot",
        "category": "label"
    }
)

encoder.fit(X_train, y_train)

X_train_enc = encoder.transform(X_train)
X_test_enc = encoder.transform(X_test)


# =================================================
# 3. SCALING (FIT ONLY ON TRAIN → NO LEAKAGE)
# =================================================

scaler = ScalingHandler(strategy="standard")

scaler.fit(X_train_enc)

X_train_scaled = scaler.transform(X_train_enc)
X_test_scaled = scaler.transform(X_test_enc)


# -------------------------
# Final Output Shapes
# -------------------------
print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)


# -------------------------
# Ready for Model Training
# -------------------------

Train shape: (302928, 55)
Test shape: (75733, 55)


In [ ]:
# # -------------------------

# # Final Processed Dataset

# # -------------------------

# final_df = pd.concat([X_encoded.reset_index(drop=True), y.reset_index(drop=True)], axis=1)

# final_df.to_csv(

#     "../data/processed/kickstarter_processed.csv",

#     index=False

# )

# print("Saved final dataset to data/processed/kickstarter_processed.csv")

# print("Shape:", final_df.shape)

Saved final dataset to data/processed/kickstarter_processed.csv
Shape: (378661, 56)


In [ ]:
# train_df = pd.concat([X_train_scaled.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)

# train_df.to_csv("data/processed/kickstarter_train_processed.csv", index=False)

In [51]:
# -------------------------
# Final Processed Dataset Save
# -------------------------

final_train_df = pd.concat(
    [
        X_train_scaled.reset_index(drop=True),
        y_train.reset_index(drop=True)
    ],
    axis=1
)

final_test_df = pd.concat(
    [
        X_test_scaled.reset_index(drop=True),
        y_test.reset_index(drop=True)
    ],
    axis=1
)

# Save files
final_train_df.to_csv(
    "../data/processed/kickstarter_train_processed.csv",
    index=False
)

final_test_df.to_csv(
    "../data/processed/kickstarter_test_processed.csv",
    index=False
)

print("Saved training set → data/processed/kickstarter_train_processed.csv")
print("Saved test set → data/processed/kickstarter_test_processed.csv")

print("\nShapes:")
print("Train:", final_train_df.shape)
print("Test:", final_test_df.shape)

Saved training set → data/processed/kickstarter_train_processed.csv
Saved test set → data/processed/kickstarter_test_processed.csv

Shapes:
Train: (302928, 56)
Test: (75733, 56)


In [53]:
print("\nMissing values (train):", final_train_df.isnull().sum().sum())

print("Missing values (test):", final_test_df.isnull().sum().sum())


Missing values (train): 0
Missing values (test): 0


In [54]:
print("\n📌 Target distribution (train):")

print(y_train.value_counts(normalize=True))

print("\n📌 Target distribution (test):")

print(y_test.value_counts(normalize=True))


📌 Target distribution (train):
state
failed        0.522154
successful    0.353761
canceled      0.102410
undefined     0.009408
live          0.007391
suspended     0.004876
Name: proportion, dtype: float64

📌 Target distribution (test):
state
failed        0.522150
successful    0.353769
canceled      0.102412
undefined     0.009401
live          0.007394
suspended     0.004872
Name: proportion, dtype: float64


In [55]:


print("\n==============================")

print("✅ DAY 2 COMPLETED SUCCESSFULLY")

print("==============================")

print("""

✔ Feature Engineering: DONE

✔ Encoding: DONE (leakage-safe)

✔ Scaling: DONE (train-only fit)

✔ Train/Test Split: DONE

✔ Processed Dataset Saved: DONE

✔ Validation Checks: PASSED

🚀 Output Ready for Day 3: Model Training

""")


✅ DAY 2 COMPLETED SUCCESSFULLY


✔ Feature Engineering: DONE

✔ Encoding: DONE (leakage-safe)

✔ Scaling: DONE (train-only fit)

✔ Train/Test Split: DONE

✔ Processed Dataset Saved: DONE

✔ Validation Checks: PASSED

🚀 Output Ready for Day 3: Model Training


